# Makespan on identical machines (chapter 3)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/mip-modelling/blob/main/notebooks/cap06_cmax.ipynb)

The second of the three problems the heuristics chapter takes up again: there the
natural order is compared with LPT on the very same four jobs. The objective is a
single variable, z, and it is the load constraints that give it meaning: this is
the min-max technique.

The full chapter — model, data, results and sensitivity analysis — is [on the website](https://fabiofurini.github.io/mip-modelling/gurobipy-4/).

## Setup

The cell below installs `gurobipy` and downloads the four shared modules of the
course: `stile.py` (palette), `mip.py` (relaxation, dual, bounds),
`euristiche.py` (next-fit, first-fit, best-fit) and `esteso.py` (the model of the
instance written out in full). The licence bundled with the pip package is limited
to **2000 variables and 2000 constraints**: the instances of the course are small
and all fit with plenty of room. For larger instances activate the free academic
licence at [portal.gurobi.com](https://portal.gurobi.com).

In [ ]:
# Environment: the solver and the shared modules of the course.
# Locally it uses the repository's python/stile.py; on Colab it installs and downloads what is missing.
import importlib.util
import subprocess
import sys
import urllib.request
from pathlib import Path

if importlib.util.find_spec("gurobipy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gurobipy", "matplotlib", "pandas", "scipy"], check=True)

for modulo in ('stile', 'mip', 'euristiche', 'esteso', 'booleane'):                     # plotting style and course utilities
    locale = next((p for p in (Path(f"../python/{modulo}.py"), Path(f"python/{modulo}.py"))
                   if p.exists()), None)
    if locale is not None:
        sys.path.insert(0, str(locale.parent.resolve()))       # notebook opened in the repository
    else:                                 # on Colab always re-download: sessions last,
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/fabiofurini/mip-modelling/main/python/{modulo}.py", f"{modulo}.py")   # modules change

In [ ]:
import gurobipy as gp
import pandas as pd
from gurobipy import GRB

from esteso import salva_modello
from mip import frazione, nuovo_modello, risolvi
from stile import intestazione, salva_dati

R = range

intestazione("Makespan: the load of the busiest machine")
d_cmax = [3, 4, 5, 6]            # job durations
k_cmax = 2                       # identical machines


def modello_cmax(d, k):
    n = len(d)
    m = nuovo_modello("makespan")
    x = m.addVars(n, k, vtype=GRB.BINARY, name="x")
    z = m.addVar(name="z")
    m.setObjective(z, GRB.MINIMIZE)
    m.addConstrs((x.sum(j, "*") == 1 for j in R(n)), name="job")
    m.addConstrs((gp.quicksum(d[j] * x[j, mm] for j in R(n)) <= z for mm in R(k)),
                 name="load")
    return m, x, z


m_cmax, x_cmax, z_var = modello_cmax(d_cmax, k_cmax)
z_cmax = risolvi(m_cmax)
salva_modello(m_cmax, "cap06_cmax")
print(f"  Durations {d_cmax} on {k_cmax} identical machines.")
print(f"  The total load is {sum(d_cmax)}: divided by {k_cmax} it gives "
      f"{frazione(sum(d_cmax) / k_cmax)}, and the optimum is {frazione(z_cmax)}.")
for mm in R(k_cmax):
    lavori = [j + 1 for j in R(len(d_cmax)) if x_cmax[j, mm].X > 0.5]
    print(f"    machine {mm + 1}: jobs {lavori}, load "
          f"{sum(d_cmax[j - 1] for j in lavori)}")
assert z_cmax == sum(d_cmax) / k_cmax
salva_dati(pd.DataFrame([{"problem": "makespan", "z_milp": z_cmax}]), "cap06_cmax")
print("Done.")

---

Notebook generated from `python/cap06_cmax.py` with `python3 python/make_notebooks.py`:
edits go into the script, not here.

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/mip-modelling/blob/main/LICENSE),
code [MIT](https://github.com/fabiofurini/mip-modelling/blob/main/LICENSE-CODE).